# Exercise 2 — Guided transfer learning with CIFAR-10

**Computer Vision for Industrial Systems**  
**Topic:** machine-learning foundations for computer vision with PyTorch

This notebook is the guided part of the exercise. It is intentionally not about inventing a new CNN architecture from scratch. Instead, you will experience a complete transfer-learning workflow:

1. load a dataset,
2. define preprocessing and augmentation,
3. adapt a pretrained CNN,
4. train for several epochs,
5. compare learning rates, epochs, architectures, and transfer strategies,
6. inspect errors.

The first dataset is **CIFAR-10**. It is not industrial, but it is small, fast, and directly available through TorchVision. That makes it a good learning dataset before we move to the industrial MVTec Capsule notebook.

## Difficulty levels

- 🟢 **Basic** — should be solved by everyone
- 🟡 **Intermediate** — requires more reasoning or minor experimentation
- 🔴 **Advanced** — optional challenge for fast students

You are expected to use the PyTorch / TorchVision documentation when needed. This is part of the exercise: in real engineering work, you will constantly check documentation.

In [ ]:
from pathlib import Path
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(repo_root / "src"))

from cvis_ml.config import DatasetConfig, ModelConfig, TrainConfig
from cvis_ml.data import CIFAR10DataModule, auto_pin_memory
from cvis_ml.models import TransferModelFactory, count_parameters, describe_trainable_parameters
from cvis_ml.engine import run_transfer_experiment, resolve_device
from cvis_ml.visualization import (
    show_batch, plot_history, show_confusion_matrix,
    show_misclassified, results_table
)

torch.manual_seed(42)
np.random.seed(42)

device = resolve_device("auto")
print("Device:", device)
print("CUDA available:", torch.cuda.is_available())
print("DataLoader pin_memory will be:", auto_pin_memory())

## Part A — CNN intuition before transfer learning

A convolutional layer applies a set of learnable filters to local image neighborhoods.

For a 2D convolution, the output height or width is:

$$
O =
\left\lfloor
\frac{I + 2P - K}{S}
\right\rfloor
+ 1
$$

where:

| Symbol | Meaning |
|---|---|
| `I` | input size |
| `P` | padding |
| `K` | kernel size |
| `S` | stride |
| `O` | output size |

We only use this to understand feature-map dimensions. We will not build a full CNN from scratch.

In [ ]:
def conv_output_size(input_size: int, kernel_size: int, stride: int = 1, padding: int = 0):
    """
    Compute the spatial output size of a convolution.

    TODO:
    1. Use the formula from the markdown cell.
    2. Return an integer.
    """
    # YOUR CODE HERE
    raise NotImplementedError("Implement conv_output_size")


# Self-check
assert conv_output_size(32, kernel_size=3, stride=1, padding=1) == 32
assert conv_output_size(32, kernel_size=3, stride=2, padding=1) == 16
print("Convolution output-size checks passed.")

In [ ]:
# Inspect a tiny convolutional layer.
conv = torch.nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, stride=1, padding=1)
x = torch.randn(4, 3, 32, 32)
y = conv(x)

print("Input shape :", tuple(x.shape))
print("Output shape:", tuple(y.shape))
print("Meaning: batch, channels, height, width")

### Short interpretation A 🟢

Answer briefly:

1. What does `out_channels=8` mean?
2. Why does padding keep the spatial size unchanged in the example?
3. Why do pretrained CNNs expect a consistent preprocessing pipeline?

In [ ]:
answer_A = """


"""
print(answer_A)

## Part B — Dataset and preprocessing

Preprocessing is part of the model contract.

The pretrained TorchVision CNNs we use were trained on ImageNet-style preprocessing. Therefore, our data module applies:

- resizing,
- optional augmentation,
- conversion to tensor,
- ImageNet normalization.

In this task, you configure a small CIFAR-10 subset. Start with the recommended values, then later you will vary them.

In [ ]:
# Configure the guided CIFAR-10 dataset.

# Recommended starting point:
# - image_size = 128
# - batch_size = 32
# - max_train_samples = 3000
# - max_val_samples = 800
# - num_workers = 0 on Windows / CPU to avoid multiprocessing issues
# - augment = True

cifar_cfg = DatasetConfig(
    name="cifar10_guided",
    data_root=str(repo_root / "data_cache"),
    image_size=...,          # TODO
    batch_size=...,          # TODO
    max_train_samples=...,   # TODO
    max_val_samples=...,     # TODO
    num_workers=...,         # TODO
    seed=42,
    augment=...,             # TODO
)

print(cifar_cfg)

In [ ]:
# Load CIFAR-10.

cifar_dm = CIFAR10DataModule(
    root=cifar_cfg.data_root,
    image_size=cifar_cfg.image_size,
    batch_size=cifar_cfg.batch_size,
    num_workers=cifar_cfg.num_workers,
    max_train_samples=cifar_cfg.max_train_samples,
    max_val_samples=cifar_cfg.max_val_samples,
    seed=cifar_cfg.seed,
    augment=cifar_cfg.augment,
)

cifar_data = cifar_dm.setup()

print("Classes:", cifar_data.class_names)
print("Number of classes:", cifar_data.num_classes)
print("Train class counts:", cifar_data.train_counts)
print("Validation class counts:", cifar_data.val_counts)
print("Train batches:", len(cifar_data.train_loader))
print("Validation batches:", len(cifar_data.val_loader))

show_batch(cifar_data.train_loader, cifar_data.class_names, n=8)

### Task B1 — Documentation lookup 🟡

Open the TorchVision transforms documentation and look up what `RandomHorizontalFlip`, `RandomCrop`, and `ColorJitter` do.

Write a short answer:

1. Which of these augmentations are plausible for CIFAR-10?
2. Which augmentations would be risky for an industrial inspection task?
3. Why is validation augmentation usually different from training augmentation?

In [ ]:
answer_B1 = """


"""
print(answer_B1)

## Part C — Transfer-learning baseline 1: frozen ResNet18

A **frozen feature extractor** means:

- the pretrained convolutional backbone is frozen,
- only the new classification head is trained.

This is the fastest and most stable transfer-learning baseline.

ResNet18 is based on residual learning: instead of learning only a direct mapping, residual blocks learn corrections through skip connections. This helps optimization in deeper CNNs.

In [ ]:
# Create a frozen ResNet18 transfer-learning model.

model_cfg = ModelConfig(
    architecture="resnet18",
    strategy="frozen",
    pretrained=True,
    num_classes=cifar_data.num_classes,
)

# TODO:
# Use TransferModelFactory.create(...) with the fields from model_cfg.
model = ...

total_params, trainable_params = count_parameters(model)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
pd.DataFrame(describe_trainable_parameters(model, max_rows=20))

## Part D — Train the first baseline

Training for only one epoch is usually not enough to observe meaningful learning curves.

For this notebook, start with:

- `epochs = 3`
- `learning_rate = 1e-3`
- `weight_decay = 1e-4`

This should still be feasible, but it produces more meaningful behavior than a one-epoch demo. If training is too slow on your machine, you may temporarily set `max_batches_per_epoch`.

In [ ]:
train_cfg = TrainConfig(
    epochs=...,              # TODO: start with 3
    learning_rate=...,       # TODO: start with 1e-3
    weight_decay=1e-4,
    device="auto",
    use_class_weights=False,
    max_batches_per_epoch=None,  # optional: set e.g. 40 if CPU is too slow
    scheduler=None,
)

result_resnet_frozen = run_transfer_experiment(
    name="cifar10_resnet18_frozen",
    model=model,
    train_loader=cifar_data.train_loader,
    val_loader=cifar_data.val_loader,
    epochs=train_cfg.epochs,
    learning_rate=train_cfg.learning_rate,
    weight_decay=train_cfg.weight_decay,
    device=train_cfg.device,
    max_batches_per_epoch=train_cfg.max_batches_per_epoch,
)

plot_history(result_resnet_frozen.history, title=result_resnet_frozen.name)
show_confusion_matrix(
    result_resnet_frozen.y_true,
    result_resnet_frozen.y_pred,
    cifar_data.class_names,
    title=result_resnet_frozen.name,
)

### Short interpretation D 🟢

Answer:

1. Did validation accuracy improve over epochs?
2. Did train loss decrease?
3. Does the model show signs of underfitting or overfitting?
4. Which classes seem most confused in the confusion matrix?

In [ ]:
answer_D = """


"""
print(answer_D)

## Part E — Learning-rate and epoch experiment 🟡

Now you design a small experiment.

You should vary:

- learning rate,
- number of epochs.

Suggested values:

| setting | epochs | learning rate |
|---|---:|---:|
| fast / maybe unstable | 3 | 1e-2 |
| recommended | 4 | 1e-3 |
| slow / conservative | 4 | 1e-4 |

In [ ]:
def run_cifar_setting(name, architecture, strategy, epochs, learning_rate, augment=True, max_batches_per_epoch=None):
    """Create data, model, and trainer for one CIFAR-10 experiment."""
    dm = CIFAR10DataModule(
        root=cifar_cfg.data_root,
        image_size=cifar_cfg.image_size,
        batch_size=cifar_cfg.batch_size,
        num_workers=cifar_cfg.num_workers,
        max_train_samples=cifar_cfg.max_train_samples,
        max_val_samples=cifar_cfg.max_val_samples,
        seed=cifar_cfg.seed,
        augment=augment,
    )
    data = dm.setup()
    model = TransferModelFactory.create(
        architecture=architecture,
        strategy=strategy,
        pretrained=True,
        num_classes=data.num_classes,
    )
    return run_transfer_experiment(
        name=name,
        model=model,
        train_loader=data.train_loader,
        val_loader=data.val_loader,
        epochs=epochs,
        learning_rate=learning_rate,
        weight_decay=1e-4,
        device="auto",
        max_batches_per_epoch=max_batches_per_epoch,
    )

In [ ]:
# TODO:
# Define at least two learning-rate / epoch settings.
# You may change the values, but explain your choice later.

experiment_settings = [
    {"name": "resnet18_frozen_lr1e-3_ep4", "architecture": "resnet18", "strategy": "frozen", "epochs": 4, "learning_rate": 1e-3, "augment": True},
    # Add at least one more setting:
    # {"name": ..., "architecture": "resnet18", "strategy": "frozen", "epochs": ..., "learning_rate": ..., "augment": True},
]

results = [result_resnet_frozen]

for cfg in experiment_settings:
    print("\nRunning", cfg)
    res = run_cifar_setting(
        name=cfg["name"],
        architecture=cfg["architecture"],
        strategy=cfg["strategy"],
        epochs=cfg["epochs"],
        learning_rate=cfg["learning_rate"],
        augment=cfg.get("augment", True),
        max_batches_per_epoch=None,
    )
    results.append(res)

results_table(results)

## Part F — Architecture and transfer-strategy experiment 🟡 / 🔴

Now compare at least two transfer-learning strategies:

1. `resnet18`, `frozen`
2. `resnet18`, `partial`
3. `mobilenet_v3_small`, `frozen`

Optional:
4. `efficientnet_b0`, `frozen`

Think about:

- validation performance,
- number of trainable parameters,
- runtime,
- stability.

In [ ]:
# TODO:
# Add at least two configurations below.
# Keep epochs moderate: 3 to 5.
# For partial fine-tuning, use a smaller learning rate such as 1e-4.

strategy_settings = [
    {"name": "resnet18_partial_lr1e-4_ep3", "architecture": "resnet18", "strategy": "partial", "epochs": 3, "learning_rate": 1e-4, "augment": True},
    {"name": "mobilenet_v3_small_frozen_lr1e-3_ep3", "architecture": "mobilenet_v3_small", "strategy": "frozen", "epochs": 3, "learning_rate": 1e-3, "augment": True},
]

for cfg in strategy_settings:
    print("\nRunning", cfg)
    res = run_cifar_setting(
        name=cfg["name"],
        architecture=cfg["architecture"],
        strategy=cfg["strategy"],
        epochs=cfg["epochs"],
        learning_rate=cfg["learning_rate"],
        augment=cfg.get("augment", True),
        max_batches_per_epoch=None,
    )
    results.append(res)

results_table(results)

## Part G — Error analysis 🟡

Metrics are useful, but they hide individual failure cases.

Choose the best result so far and inspect misclassified validation examples.

In [ ]:
# TODO:
# Choose which trained model to inspect.
# The variable `model` still contains the first frozen ResNet18 model.
# If you want to inspect a later model, rerun that model or adapt the helper above to return it.

show_misclassified(
    model,
    cifar_data.val_loader,
    cifar_data.class_names,
    device=resolve_device("auto"),
    n=8,
    max_batches=20,
)

### Final CIFAR-10 mini-report 🟡

Write 6–10 sentences:

1. Which transfer-learning baseline performed best?
2. Which learning rate was most stable?
3. Did more epochs help?
4. Did augmentation help or hurt?
5. Which architecture/strategy gave the best trade-off between performance and runtime?
6. What would you try next if you had more compute?

In [ ]:
final_cifar_report = """


"""
print(final_cifar_report)